%pip install pandas

In [4]:
import json
import glob
import pandas as pd

# Auto-detect latest files
jsonl_files = sorted(glob.glob('./debug_*.jsonl'))
part_files = sorted(glob.glob('./submission_*.part'))

JSONL_PATH = jsonl_files[-1] if jsonl_files else None
PART_PATH = part_files[-1] if part_files else None

print('JSONL:', JSONL_PATH)
print('PART: ', PART_PATH)

JSONL: ./debug_qwen3b_svg_lora_1774726326.jsonl
PART:  ./submission_qwen3b_svg_lora_1774726326.csv.part


In [ ]:
# CSV.part progress 
part_df = pd.read_csv(PART_PATH)
total_so_far = len(part_df)

fallback_svg = ('<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
                '<rect x="0" y="0" width="256" height="256" fill="white"/>'
                '<circle cx="128" cy="128" r="64" fill="black"/>'
                '</svg>')

fallback_count = (part_df['svg'] == fallback_svg).sum()
fallback_pct = 100 * fallback_count / total_so_far if total_so_far else 0

print(f'Rows written so far : {total_so_far}')
print(f'Fallback SVGs       : {fallback_count}  ({fallback_pct:.1f}%)')
print(f'Real SVGs           : {total_so_far - fallback_count}  ({100 - fallback_pct:.1f}%)')

Rows written so far : 512
Fallback SVGs       : 166  (32.4%)
Real SVGs           : 346  (67.6%)


In [ ]:
# JSONL failure breakdown 
failures = []
with open(JSONL_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            failures.append(json.loads(line))

fail_df = pd.DataFrame(failures)
print(f'Failure entries in JSONL: {len(fail_df)}')

if len(fail_df):
    print()
    print('Breakdown:')
    print(f'  no_svg_found  : {fail_df["no_svg_found"].sum()}')
    print(f'  parse_error   : {fail_df["parse_error"].notna().sum()}')
    print()
    print('Sample failure prompts:')
    for row in fail_df.head(5).itertuples():
        print(f'  [{row.Index}] {row.prompt}')
        if row.parse_error:
            print(f'       parse_error: {row.parse_error}')
        print(f'       raw (first 200): {row.raw_decoded[:200]}')
        print()

Failure entries in JSONL: 166

Breakdown:
  no_svg_found  : 166
  parse_error   : 0

Sample failure prompts:
  [0] A pair of men's underwear with a contour designed to cradle the natural shape of the wearer's lower body.
       raw (first 200): <svg height="128" viewBox="-3 0 56 56" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m47.9 1h-4a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1

  [1] A single, dark gray outline of an abstract shape resembling a leaf or flame against a white background.
       raw (first 200): <svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#47516E" fill-opacity="1.0"  filling="0" d="M98.0 12.0 C100.0 12.0 101.0 12.0 102.0 1

  [2] A cartoon illustration of a split pea pod with two peas inside.
       raw (first 200): <svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#F4D891"

In [7]:

import re

SVG_OPEN_RE = re.compile(r'<svg[\s\S]*', re.IGNORECASE)
SVG_CLOSE_RE = re.compile(r'</svg>', re.IGNORECASE)

# Categorise each failure
cats = []
for entry in failures:
    raw = entry.get('raw_decoded', '')
    has_open = bool(re.search(r'<svg', raw, re.IGNORECASE))
    has_close = bool(SVG_CLOSE_RE.search(raw))
    parse_err = entry.get('parse_error')

    if not has_open:
        cat = 'no_svg_tag'          # model never emitted <svg at all
    elif not has_close:
        cat = 'truncated_no_close'  # hit token limit mid-SVG (repetition loop)
    elif parse_err:
        cat = 'parse_error'         # complete tag but malformed XML
    else:
        cat = 'other'
    cats.append(cat)

fail_df['category'] = cats

print('=== Failure categories ===')
print(fail_df['category'].value_counts().to_string())
print()

# For truncated ones, check if the d="" path attribute is looping
truncated = fail_df[fail_df['category'] == 'truncated_no_close']
print(f'\nTruncated (hit token limit): {len(truncated)}')
if len(truncated):
    # Measure raw length as proxy for whether 2000 tokens were used
    truncated = truncated.copy()
    truncated['raw_len'] = truncated['raw_decoded'].str.len()
    print(f'  Avg raw chars  : {truncated["raw_len"].mean():.0f}')
    print(f'  Max raw chars  : {truncated["raw_len"].max()}')
    print()
    # Check for coordinate-fragment repetition in path d= attributes
    def detect_repetition(raw, window=20, threshold=10):
        """Return the repeated fragment if a short substring repeats >= threshold times."""
        for length in range(8, window):
            for start in range(0, min(200, len(raw) - length * threshold)):
                frag = raw[start:start + length]
                if raw.count(frag) >= threshold:
                    return frag
        return None

    reps = truncated['raw_decoded'].apply(detect_repetition)
    n_rep = reps.notna().sum()
    print(f'  Repetition loop detected in {n_rep}/{len(truncated)} truncated entries')
    if n_rep:
        example_frag = reps.dropna().iloc[0]
        print(f'  Example repeated fragment: {repr(example_frag[:60])}')


=== Failure categories ===
category
truncated_no_close    166


Truncated (hit token limit): 166
  Avg raw chars  : 2000
  Max raw chars  : 2000

  Repetition loop detected in 124/166 truncated entries
  Example repeated fragment: 'a1 1 0 0'


In [ ]:
# Show a representative sample of raw outputs for each category
for cat in fail_df['category'].unique():
    subset = fail_df[fail_df['category'] == cat]
    print(f'\n{"="*60}')
    print(f'Category: {cat}  ({len(subset)} entries)')
    print('='*60)
    for _, row in subset.head(3).iterrows():
        print(f'  PROMPT : {row["prompt"][:100]}')
        print(f'  RAW    : {repr(row["raw_decoded"][:300])}')
        if row.get('parse_error'):
            print(f'  ERROR  : {row["parse_error"]}')
        print()



Category: truncated_no_close  (166 entries)
  PROMPT : A pair of men's underwear with a contour designed to cradle the natural shape of the wearer's lower 
  RAW    : '<svg height="128" viewBox="-3 0 56 56" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m47.9 1h-4a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1v1h-1a1 1 0 0 0 -1 1'

  PROMPT : A single, dark gray outline of an abstract shape resembling a leaf or flame against a white backgrou
  RAW    : '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#47516E" fill-opacity="1.0"  filling="0" d="M98.0 12.0 C100.0 12.0 101.0 12.0 102.0 13.0 L102.0 13.0 A10.0 10.0 0.0 0 1 112.0 23.0 L112.0 23.0 A10.0 10.0 0.0 0 1 122.0 33.0 L122.0 33.0 '

  PROMPT : A cartoon illustration of a split pea pod with two peas inside.
  RAW    : '<svg xmlns="ht